# صوت الفيديو الترويجي بـ ElevenLabs
شغّل الخلايا بالترتيب.
1. **الخلية 2:** ضع مفتاح API في أسرار Colab (أيقونة المفتاح 🔑 على اليسار) باسم `ELEVEN_API_KEY`. **لا ترسل المفتاح لأحد.**
2. **الخلية 4:** تعرض الأصوات المتاحة في حسابك. انسخ `voice_id` للصوت الذي يعجبك في `VOICE_ID`.
3. **الخلية 5:** تجربة جملة واحدة.
4. **الخلية 6:** الصوت كاملًا. كل جملة تُحفظ وحدها، فإن أعدت التشغيل لا تُولَّد إلا الجمل المذكورة في `REDO` أو التي لم تُولَّد بعد.
5. أرسل لي `promo2_voice.zip` كما هو.


In [ ]:
!pip -q install requests pydub "audioop-lts; python_version>='3.13'"


In [ ]:
from google.colab import userdata
API_KEY = userdata.get("ELEVEN_API_KEY")
MODEL = "eleven_multilingual_v2"
VOICE_ID = ""          # ضع هنا voice_id من الخلية 4
SETTINGS = {"stability": 0.55, "similarity_boost": 0.8, "style": 0.15, "use_speaker_boost": True, "speed": 1.0}
GAP = 0.6              # صمت بين الجمل (ثوانٍ)
LEAD = 0.4
REDO = []              # أرقام الجمل التي تريد إعادة توليدها (تبدأ من 1)، مثلًا [3, 7]
LINES = [
 "لكل أستاذ تعليم ثانوي يدرّس في القسم أو في دروس الدعم، ولكل تلميذ في الثانوي، خاصة تلاميذ البكالوريا.",
 "هل تبحث عن مكتبة رياضيات كاملة فيها كل ما تحتاج؟ ملخصات، ومذكرات، وسلاسل تمارين نموذجية مرفقة بالحل المفصل، وجميع تمارين البكالوريا بالحل المفصل.",
 "إليك مكتبة الأستاذ سوايسية هشام: أكثر من مئة وأربعين ملفًا، مطابقة للمنهاج الجزائري وللتوزيع السنوي الجديد.",
 "للسنة الثالثة ثانوي: ملخص شامل لكل محور، وسلسلة تمارين نموذجية بحلول مفصلة، وحلول تمارين البكالوريا لشعب العلوم التجريبية والرياضيات والتقني رياضي، وشعبة التسيير والاقتصاد.",
 "نبدأ بالسنة الثالثة ثانوي، الشعب العلمية.",
 "الدالة الأسية: ملخص في ثمان وثلاثين صفحة بالأمثلة والمنحنيات، وسلسلة تمارين نموذجية محلولة، وتمارين البكالوريا بالحل المفصل، بصيغ بي دي إف ووورد ولاتاك.",
 "الدالة اللوغاريتمية: ملخص، وسلسلة تمارين، وأكثر من مئة وسبعين صفحة من تمارين البكالوريا المحلولة.",
 "الدوال الأصلية والتكامل: حساب المساحات مع تلوين الحيز في كل رسم، وتمارين البكالوريا محلولة.",
 "الاحتمالات: أشجار منسقة، وجداول قوانين الاحتمال، وتمارين البكالوريا بالحل المفصل.",
 "وبنفس الطريقة: النهايات، والمتتاليات، والأعداد المركبة، والحساب، والهندسة في الفضاء، ومذكرات كل المحاور.",
 "شعبة التسيير والاقتصاد: المتتاليات العددية، بملخص وتمارين البكالوريا المحلولة.",
 "والدوال الأصلية والتكامل، بحساب المساحات وتمارين البكالوريا.",
 "والدالة اللوغاريتمية، بملخص وسلسلة تمارين محلولة.",
 "ننتقل إلى السنة الثانية ثانوي، الشعب العلمية: الاشتقاقية، سلسلة تمارين محلولة خطوة بخطوة.",
 "والمرجح، والاحتمالات، والنهايات: ملخصات وتمارين بالحلول المفصلة.",
 "ثم السنة الأولى ثانوي، جذع مشترك علوم وتكنولوجيا: الدوال المرجعية، والحساب الشعاعي، والهندسة المستوية، بملخصات وسلاسل تمارين محلولة.",
 "أستاذ: ملفات تساعدك في إنجاز الفروض والاختبارات وسلاسل الدعم. وتلميذ: راجع بثقة، فكل حل مكتوب خطوة بخطوة.",
 "وكل ملف بثلاث صيغ: بي دي إف للطباعة، ووورد ولاتاك للتعديل عليها كما تشاء.",
 "للحصول على جميع الملفات، أرسل رسالة في الخاص على صفحة الأستاذ سوايسية هشام للرياضيات، أو على واتساب عبر هذا الرقم."
]
# تصحيح النطق: الكلمة كما تظهر على الشاشة -> الكتابة التي تُنطق (تشكيل أو كتابة صوتية).
# كلمة واحدة مقابل كلمة واحدة حتى يبقى توقيت الكتابة صحيحًا.
FIX = {
 "لاتاك": "لاتِك",
 "وورد": "وُورْد",
}
def spoken(line):
    return " ".join(FIX.get(w.strip("،:؟.!"), w.strip("،:؟.!")) + w[len(w.rstrip("،:؟.!")):] if w.strip("،:؟.!") in FIX else w for w in line.split())


In [ ]:
import requests
r = requests.get("https://api.elevenlabs.io/v1/voices", headers={"xi-api-key": API_KEY}).json()
for v in r.get("voices", []):
    print(v["voice_id"], "|", v["name"], "|", v.get("labels", {}))


In [ ]:
import base64, io, os, json
from pydub import AudioSegment
from IPython.display import Audio, display
os.makedirs("parts", exist_ok=True)

def tts(text):
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}/with-timestamps?output_format=mp3_44100_128"
    r = requests.post(url, headers={"xi-api-key": API_KEY}, json={"text": text, "model_id": MODEL, "voice_settings": SETTINGS})
    r.raise_for_status(); j = r.json()
    seg = AudioSegment.from_file(io.BytesIO(base64.b64decode(j["audio_base64"])), format="mp3")
    a = j["alignment"]; words, cur = [], None
    for ch, s, e in zip(a["characters"], a["character_start_times_seconds"], a["character_end_times_seconds"]):
        if ch.isspace():
            if cur: words.append(cur); cur = None
        elif cur is None: cur = {"text": ch, "start": s, "end": e}
        else: cur["text"] += ch; cur["end"] = e
    if cur: words.append(cur)
    return seg, words

seg, w = tts(spoken(LINES[0]))
print(len(w), "كلمة"); display(Audio(seg.export(format="mp3").read()))


In [ ]:
import zipfile
from google.colab import files
full = AudioSegment.silent(duration=int(LEAD * 1000)); out = []
for i, line in enumerate(LINES, 1):
    mp3, js = f"parts/{i:02d}.mp3", f"parts/{i:02d}.json"
    if i in REDO or not os.path.exists(mp3):
        seg, words = tts(spoken(line)); seg.export(mp3, format="mp3", bitrate="192k")
        json.dump(words, open(js, "w"), ensure_ascii=False)
        print(i, "توليد", round(len(seg) / 1000, 1), "ث")
    seg = AudioSegment.from_file(mp3); words = json.load(open(js))
    shown = line.split()
    if len(shown) == len(words):            # نعرض الكلمات كما في السيناريو (قبل تصحيح النطق)
        for w_, s_ in zip(words, shown): w_["text"] = s_
    t0 = len(full) / 1000
    out.append({"i": i, "text": line, "start": t0, "end": t0 + len(seg) / 1000,
                "words": [{**w_, "start": w_["start"] + t0, "end": w_["end"] + t0} for w_ in words]})
    full += seg + AudioSegment.silent(duration=int(GAP * 1000))
full.export("voice.mp3", format="mp3", bitrate="192k")
json.dump({"voice": VOICE_ID, "model": MODEL, "duration": len(full) / 1000, "lines": out}, open("timings.json", "w"), ensure_ascii=False, indent=1)
with zipfile.ZipFile("promo2_voice.zip", "w") as z:
    z.write("voice.mp3"); z.write("timings.json")
print("المدة:", round(len(full) / 1000), "ثانية"); display(Audio("voice.mp3"))
files.download("promo2_voice.zip")
